<a href="https://colab.research.google.com/github/shown5/Hands-on-Generative-AI/blob/main/chap8_advanced_use.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

8.1 画像から画像

Stable Diffusion で学んだように、画像にノイズを加え、それを除去する過程を経てテキストから画像を生成していた。
これをテキストなしに画像から始めることを Picture to Picture とよぶ。
diffuser ライブラリを使えば画像から画像の生成を試すことができる。

In [1]:
%pip install genaibook

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 61.7 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
  Attempting uninstall: huggingface_hub
    Found exi

In [ ]:
import torch
from diffusers import StableDiffusionXLImg2ImgPipeline
from genaibook.core import get_device

device = get_device()

# パイプラインを読み込む
img2img_pipeline = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-refiner-1.0",
    torch_dtype=torch.float32, # Changed from torch.float16 to torch.float32
    variant="fp16", # variant might need to be adjusted or removed if fp16 specific
)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


model_index.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

unet/diffusion_pytorch_model.fp16.safete(…):   0%|          | 0.00/4.52G [00:00<?, ?B/s]

vae/diffusion_pytorch_model.fp16.safeten(…):   0%|          | 0.00/167M [00:00<?, ?B/s]

vae_1_0/diffusion_pytorch_model.fp16.saf(…):   0%|          | 0.00/167M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

次にパイプラインをデバイスに移動させる（どういうこと？）


In [ ]:
img2img_pipeline.to(device)

これで例が整ったので実際に試してみる。

In [ ]:
from genaibook.core import SampleURL, load_image, image_grid

#画像を読み込む
url = SampleURL.ToyAstronauts
init_image = load_image(url)

prompt = "Astronaut in a jungle, cold color palette, muted colors, detailed, 8k"

# 画像とプロンプトをパイプラインに渡す
image = img2img_pipeline(prompt, image=init_image, strength=0.5).images[0]
image_grid([init_image, image], rows=1, cols=2)

8.2 インペインティング

従来のインペインティング方式は対象領域をさまざまな方法でマスクする方法が取られてきた。
生成的なインペインティングでは、視覚的・意味的な文脈の両方を理解し、それに基づいて新たな内容を生成できる。

In [ ]:
from diffusers import StableDiffuisionXLInpaintPipeline

# パイプラインを読み込む
inpaint_pipeline = StableDiffusionXLInpaontPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16"
).to(device)

img_url = SampleURL.DogBenchImage
mask_url = SampleURL.DogBenchMask

init_image = load_image(img_url).convert("RGB").resize((1024, 1024))
mask_image = load_image(mask_url).convert("RGB").resize((1024, 1024))

# プロンプトと画像をパイプラインに渡す
prompt = "A majestic tiger sitting on a bench"
image = inpaint_pipeline(
    prompt=prompt,
    image=init_image,
    mask_image=mask_image,
    num_inference_steps=50,
    strength=0.80,
    width=init_image.size[0],
    height=init_image.size[1]
).images[0]

In [ ]:
image_grid([init_image, mask_image, image], rows=1, cols=3)

8.3 プロンプト重みづけと画像編集

拡散モデルは transformer に似た attention 機構を用いているため入力の重要な部分に対して柔軟に注目できるようになっている。
プロンプトの単語の重みを調整したり、複数のプロンプトを組み合わせて画像を生成したり、画像編集の際に構造を保ったまま生成結果の身を変えたいといった、細かなニーズにも対応できる。
本節ではこれらの細かな方法についてみていく。

8.3.1 プロンプト重みづけとマージ

プロンプトの重みづけには、compel プロンプト拡張ライブラリが使える。
文字列を前処理し、CLIP 埋め込み空間内に対応する埋め込みを強調する形で動作する。

- 単語の後ろに＋をつけて重みを高められる。ーをつければ低められる。
- ２つのプロンプトを角カッコ内に配置して、プロンプト同士をマージすることで、それぞれのプロンプトに重みを指定する

以下に簡単なコード

In [ ]:
from diffusers import DiffusionPipeline

pipeline = DiffusionPipeline.from_pretrained(
    "stablityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
).to(device)

In [ ]:
# compel クラスを初期化する

from compel import Compel, RetrunedEmbeddingsType

#CLIP モデルで最も表現力が高いとされる最終層から２番目の層を使用する
embedding_type = (
    RetrunedEmbeddingsType.PENULTIMATE_HIDDEN_STATES_NON_NORMALIZED
)
compel = Comp(
    tokenizer=[pipeline.tokenizer, pipeline.tokenizer_2],
    text_encoder=[pipeline.text_encoder, pipeline.text_encoder_2],
    returned_embeddings_type=embeddings_type,
    reuires_pooled=[False, True]
)

In [ ]:
from genaibook.core import image_grid

# プロンプトの準備
prompts = []
prompts.append("a humanoid robot eating pasta")
prompts.append(
    "a humanoid+++ robot eating pasta"
) #人型ロボットの特徴を少し強調する
prompts.append(
    '["a humanoid robot eating pasta", "a van gogh painting"].and(0.8, 0.2)'
) #ゴッホ風にする

images = []
for prompts in prompts:
  #全ての生成で同じシード値を使用する
  generator = torch.Generator(device=device).manual_seed(1)

  #compel ライブラリは条件付けのベクトルとプーリングプロンプトの埋め込みの両方を返す
  conditioning, pooled = compel(prompt)

  # 条件とプーリングプロンプトの埋め込みをパイプラインに渡す
  image = pipeline(
      prompt_embeds=conditioning,
      pooled_prompt_embeds=pooled,
      num_inference_steps=30,
      generator=generator,
  ).images[0]
  images.append(image)
image_grid(images, rows=1, cols=3)

8.3.2 Semantic Guidance による拡散画像の編集

Sematic Guidance（SEGA） も画像編集の一つの方法。
逆拡散プロセスの各ステップでモデルのノイズ推定値を操作する形で動作する。
テキスト埋め込みと潜在空間との間の勾配を計算し、画像生成や編集を望みの意味的結果へと導く。

⭐️全然わからない

In [ ]:
from diffusers import SemanticStableDiffusionPipeline

semantic_pipeline = SemanticStableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4", torch_dtype=torch.float16, variant="fp16"
).to(device)

In [ ]:
generator = torch.Generator(device=device).manual_seed(100)
out = semantic_pipeline(
    prompt="a photo of the face of a man",
    negative_prompt="low quality, deformed",
    generator=generator
)
out.images[0]

In [ ]:
# 男性を笑顔にする方向にプロンプトを誘導する

generator = torch.Generator(device=device).manual_seed(100)
out = semantic_pipeline(
    prompt="a photo of the face of a man",
    negative_prompt="low quality, deformed",
    editing_prompt="smiling, smile",
    edit_guidance_scale=4,
    edit_warmup_steps=10,
    edit_threshold=0.99,
    edit_momentum_scale=0.3,
    edit_mom_beta=0.6,
    reverse_editing_direction=False,
    generator=generator
)
out.images[0]

In [ ]:
# メガネをかけさせる

generator = torch.Generator(device=device).manual_seed(100)
out = semantic_pipeline(
    prompt="a photo of the face of a man",
    negative_prompt="low quality, deformed",
    editing_prompt="glasses, wearing glasses",
    edit_guidance_scale=4,
    edit_warmup_steps=10,
    edit_threshold=0.99,
    edit_momentum_scale=0.3,
    edit_mom_beta=0.6,
    reverse_editing_direction=False,
    generator=generator
)
out.images[0]

In [ ]:
# 複数の編集を同時に適用

generator = torch.Generator(device=device).manual_seed(100)
out = semantic_pipeline(
    prompt="a photo of the face of a man",
    negative_prompt="low quality, deformed",
    editing_prompt=[
        "smiling, smile",
        "glasses, wearing glasses",
    ]
    edit_guidance_scale=[4, 4]
    edit_warmup_steps=[10, 10]
    edit_threshold=[0.99, 0.99]
    edit_momentum_scale=0.3,
    edit_mom_beta=0.6,
    reverse_editing_direction=[False, False]
    generator=generator
)
out.images[0]

8.4 インバージョンによる実画像編集

インバージョンとは実際の画像を事前訓練済み生成モデルの潜在空間へと戻す手法。
通常、生成モデルは潜在空間から画像を生成するが、インバージョンはその逆で、画像を潜在空間に表現すること。